# PA 766 | 2. Turn text into a CSV for annotation

**Goal:** divide the text files from Notebook 1 into manageable chunks and export one CSV for manual annotation.

Open this notebook in **Google Colab** and run the cells in order. Upload `PA766_text_files.zip` from Notebook 1, or its individual `.txt` files. This notebook works in a new Colab session and needs no API key.

We will practice one dimension from the CARESL annotation guide: **Statement status**. The question is: **How does this passage frame a concrete action for managing electricity demand or load growth?** LLM annotation is a later step and is not part of this notebook.

## 1. Upload your text files

Select the ZIP from Notebook 1, or select several `.txt` files together. Upload each document once. The ZIP is read directly; you do not need to unzip it in Colab.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import pandas as pd
from google.colab import files
from IPython.display import display

uploaded = files.upload()
texts = {}

def add_text(name, data):
    name = Path(name).name
    if name in texts:
        raise ValueError(f"Duplicate file: {name}. Upload each document only once.")
    texts[name] = data.decode("utf-8-sig")

for filename, data in uploaded.items():
    if filename.lower().endswith(".txt"):
        add_text(filename, data)
    elif filename.lower().endswith(".zip"):
        with ZipFile(BytesIO(data)) as archive:
            for member in archive.infolist():
                if not member.is_dir() and member.filename.lower().endswith(".txt"):
                    add_text(member.filename, archive.read(member))

if not texts:
    raise ValueError("Upload .txt files or the ZIP produced by Notebook 1.")
print(f"Loaded {len(texts)} text file(s):")
for name in sorted(texts):
    print(name)

## 2. Choose a chunk size

We use a simple rule: **combine consecutive text blocks up to 180 words, staying within a PDF page**. A block longer than the limit is split into word groups. Chunks do not overlap, and all extracted words are retained in their existing order.

This is an approximate reading length, not a token limit or a guarantee of a complete idea. A long block may be split mid-sentence, and a passage may continue on the next page. Headings, references, and table fragments can become chunks too. Inspect the text before assigning a label.

Start with 180. You can try another size before annotation begins. Changing the input text or chunk size changes the rows and their IDs; keep the original CSV once you begin labeling.

In [ ]:
MAX_WORDS = 180
if not isinstance(MAX_WORDS, int) or MAX_WORDS < 1:
    raise ValueError("MAX_WORDS must be a positive whole number.")

def chunk_page(page_text, max_words):
    chunks = []
    current = []
    blocks = page_text.split("\n\n")
    for block in blocks:
        words = block.split()
        if not words:
            continue
        if current and len(current) + len(words) > max_words:
            chunks.append(" ".join(current))
            current = []
        while len(words) > max_words:
            chunks.append(" ".join(words[:max_words]))
            words = words[max_words:]
        current.extend(words)
    if current:
        chunks.append(" ".join(current))
    return chunks

## 3. Create the dataset

Each row is one chunk. We retain the source filename and the PDF page number so you can return to the original. `statement_status` and `notes` start blank; you will fill them manually in a spreadsheet.

Page numbers depend on the `\f` page separators written by Notebook 1. Keep those files unchanged. Ordinary text files without these separators will be treated as one page.

In [ ]:
rows = []
empty_pages = []
for filename, text in sorted(texts.items()):
    for page_number, page_text in enumerate(text.split("\f"), start=1):
        chunks = chunk_page(page_text, MAX_WORDS)
        if not chunks:
            empty_pages.append(f"{filename}: page {page_number}")
        for chunk_number, chunk in enumerate(chunks, start=1):
            rows.append({
                "chunk_id": f"{Path(filename).stem}_p{page_number:03d}_c{chunk_number:02d}",
                "source_file": filename,
                "page": page_number,
                "text": chunk,
                "word_count": len(chunk.split()),
                "statement_status": "",
                "notes": "",
            })

if not rows:
    raise ValueError("The uploaded files contain no usable text.")
dataset = pd.DataFrame(rows)
assert dataset["chunk_id"].is_unique
assert dataset["word_count"].between(1, MAX_WORDS).all()
print(f"Created {len(dataset)} chunks from {len(texts)} document(s).")
display(dataset.groupby("source_file", as_index=False).agg(chunks=("chunk_id", "count")))
if empty_pages:
    print("No text on these pages; no CSV row was created:")
    print("\n".join(empty_pages))
display(dataset.head(5))

## 4. Inspect a chunk

Change `ROW_NUMBER` to inspect a different row. The first row is numbered 0 in Python. Use `source_file` and `page` to find it in the PDF. A cover or contents page is not necessarily a useful practice passage.

In [ ]:
ROW_NUMBER = 0
row = dataset.iloc[ROW_NUMBER]
print(row["chunk_id"])
print(f"Source: {row['source_file']} | PDF page: {row['page']} | Words: {row['word_count']}\n")
print(row["text"])

## 5. Use one annotation dimension: Statement status

First look for a **concrete action intended to manage, accommodate, or respond to electricity demand or load growth**. A forecast, risk, broad goal, or technology name alone is not an action. Code what the passage says, without checking whether the action happened in the real world.

Enter one of these exact codes in `statement_status`:

| Code | Meaning from the guide | Invented practice example |
|---|---|---|
| `recommendation` | The passage advises or recommends an action. | Utilities should offer managed charging programs. |
| `plan` | An actor intends or commits to a future action. | The utility plans to launch a managed charging program next year. |
| `practice` | An action is underway, operating, or completed. | The utility launched its managed charging program last year. |
| `general` | The action is described as general, hypothetical, conditional, or of unclear status. | A managed charging program could reduce evening demand. |

Use two additional **classroom handling codes** when a row cannot receive one of those four labels:

- `no_action`: no concrete load-management action appears in the chunk. Examples include a cover, references, or a load forecast alone.
- `review`: the text is damaged, needed context is missing, or multiple actions/statuses prevent a single clear label. Briefly explain in `notes`. Do not guess from words such as “will” or “should” alone.

Several actions with the same clear status can receive that status. When statuses differ, use `review`. For `general`, a recognizable action must still be present; use `review` for unreadable or insufficient evidence.

**Classroom adaptation:** The source is CARESL's `annotation_specs/v0.1/annotation-manual.md`, section “Statement status.” The research guide allows multiple statuses on a strategy instance. Here we label whole chunks with one code and use `review` for mixed cases. The two handling codes are teaching additions, not original status categories. Chunk counts are not counts of distinct strategies.

Start with **10 instructor-assigned chunks**. Two students can independently annotate the same chunk IDs and discuss differences. Leave unreviewed rows blank. Keep the `text`, IDs, and source fields unchanged; write comments in `notes`. If checking the original resolves an ambiguity, record the context you used in `notes`.

## 6. Download and annotate the CSV

The CSV contains all chunks with blank labels. Open it in Google Sheets using **File > Import > Upload**, or in Excel. Turn on text wrapping and freeze the header row. Edit only `statement_status` and `notes`. The file uses UTF-8 with a BOM to help Excel preserve punctuation.

Save your annotated copy under a new name, such as `PA766_annotations_yourname.csv`. In Google Sheets, use **File > Download > Comma-separated values (.csv)**. Re-running this notebook creates a new blank template; it does not read or preserve labels entered in your spreadsheet.

In [ ]:
csv_path = Path("PA766_chunks_for_annotation.csv")
dataset.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(dataset)} rows to {csv_path.name}")
files.download(str(csv_path))

### Discuss

1. Find a chunk that is easy to label and explain which wording supports your choice.
2. Find a chunk that needs more context. Would a different boundary help?
3. Why should a blank label mean “not yet reviewed,” rather than “no action”?

Keep the CSV and the original PDFs. The dataset can later be used for LLM annotation, but this activity ends with preparing the dataset and practicing manual annotation.